# 03 — Feature Engineering & Menyimpan Dataset

**Tujuan:** menambah kolom turunan yang **membuat analisis dan dashboard jauh lebih
kaya**, lalu menyimpannya dalam dua format:

- **CSV** — portabel, bisa dibuka di Excel, enak dibagikan
- **joblib** — ringkas dan cepat dimuat ulang, menjaga tipe data (kategori, list) →
  inilah yang dipakai dashboard

Kenapa perlu fitur turunan? Data mentah punya sedikit kolom numerik; dengan fitur
seperti `decade`, `mood`, `duration_band`, analisis jadi jauh lebih komunikatif —
misalnya "seberapa banyak lagu Melankolis tiap dekade" langsung bisa dihitung tanpa
transformasi ulang.

## Fitur yang dibuat

**Per lagu:**

| Fitur | Asal | Kegunaan |
|---|---|---|
| `key_name` | `key` (0–11) | nama nada: C, C#, D … |
| `mode_name` | `mode` (0/1) | Major / Minor |
| `decade` | `year` | tren per dekade |
| `duration_band` | `duration_min` | 4 kategori panjang lagu |
| `tempo_band` | `tempo` | santai / sedang / cepat |
| `energy_band` | `energy` | rendah / sedang / tinggi |
| `mood` | `valence` × `energy` | Mood/Vibe lagu → Emosi (highlight dashboard) |
| `is_outlier` | IQR | penanda (bukan penghapusan) nilai ekstrem |

**Tabel turunan:** `genres_long` (1 lagu × 1 genre) dan `artists` (agregat per artis).

## 1. Muat hasil cleaning

In [1]:
import numpy as np
import pandas as pd
import os
import joblib

CLEAN_IN = "../data/clean/songs_clean.csv"     # hasil notebook 02
SONGS_OUT = "../data/clean/songs_final.csv"    # hasil notebook 03 (fitur lengkap)
GENRES_OUT = "../data/clean/genres_long.csv"
ARTISTS_OUT = "../data/clean/artists.csv"
JOBLIB_OUT = "../data/clean/spotify_clean.joblib"   # sumber tunggal dashboard

songs = pd.read_csv(CLEAN_IN)
print("Dimuat", CLEAN_IN, "->", songs.shape)

# Penting: CSV tidak punya tipe list, jadi kolom genre_list terbaca sebagai TEKS
# ("['hip hop', 'pop']"). Bangun ulang sebagai list sungguhan, kalau tidak filter
# multi-genre di dashboard tidak akan pernah cocok.
songs["genre_list"] = songs["genre"].apply(
    lambda g: [bagian.strip() for bagian in str(g).split(",") if bagian.strip()]
)
print("Tipe isi genre_list:", type(songs["genre_list"].iloc[0]).__name__)

Dimuat ../data/clean/songs_clean.csv -> (1926, 23)
Tipe isi genre_list: list


## 2. Ubah kode musik jadi nama yang bisa dibaca

`key` tetap berguna sebagai angka, tapi `key_name` ("C", "F#", …) membuat grafik
bisa dibaca orang awam. `mode_name` menerjemahkan 0 → Minor, 1 → Major.

In [2]:
KEY_NAMES = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]

songs["key_name"] = songs["key"].map(lambda k: KEY_NAMES[k])
songs["mode_name"] = songs["mode"].map({0: "Minor", 1: "Major"})

# Nada paling sering di lagu-lagu populer?
print("Top 5 nada (key_name):")
print(songs["key_name"].value_counts().head(5).to_string())
print("\nProporsi Major/Minor:")
print(songs["mode_name"].value_counts(normalize=True).round(3).to_string())

Top 5 nada (key_name):
key_name
C#    255
C     194
B     191
G     189
F     174

Proporsi Major/Minor:
mode_name
Major    0.553
Minor    0.447


## 3. `decade` — kelompokkan tahun untuk tren

Data berisi 1998–2020. Mengelompokkan per dekade membuat tren lebih terbaca
daripada 23 garis tahun individual.

In [3]:
decade_str = (songs["year"] // 10 * 10).astype(str) + "an"   # 1998 -> "1990an"

# Ordered categorical supaya urutan plot selalu benar (bukan alfabetis)
DECADE_ORDER = ["1990an", "2000an", "2010an", "2020an"]
songs["decade"] = pd.Categorical(decade_str, categories=DECADE_ORDER, ordered=True)

print(songs["decade"].value_counts().sort_index().to_string())

decade
1990an     39
2000an    904
2010an    980
2020an      3


## 4. Band: `duration_band`, `tempo_band`, `energy_band`

Orang tidak mudah membaca angka kontinu ("272 detik", "0.83 energy"), tapi mudah
membaca kategori ("3:30–4:30", "cepat", "tinggi"). Batas dipilih berdasarkan
konvensi musik/BPM yang lazim, bukan angka ajaib.

In [4]:
def duration_band(menit):
    if menit < 3.5: return "< 3:30"
    if menit < 4.5: return "3:30-4:30"
    if menit < 5.5: return "4:30-5:30"
    return "> 5:30"

def tempo_band(bpm):
    if bpm < 90:  return "Santai (<90 BPM)"
    if bpm < 120: return "Sedang (90-120 BPM)"
    if bpm < 150: return "Cepat (120-150 BPM)"
    return "Sangat cepat (>=150 BPM)"

def energy_band(energy):
    if energy < 0.4: return "Rendah (<0.4)"
    if energy < 0.7: return "Sedang (0.4-0.7)"
    return "Tinggi (>=0.7)"

# Semua dikemas sebagai kategori berurutan agar grafik tersusun rapi
DURATION_ORDER = ["< 3:30", "3:30-4:30", "4:30-5:30", "> 5:30"]
TEMPO_ORDER = ["Santai (<90 BPM)", "Sedang (90-120 BPM)", "Cepat (120-150 BPM)", "Sangat cepat (>=150 BPM)"]
ENERGY_ORDER = ["Rendah (<0.4)", "Sedang (0.4-0.7)", "Tinggi (>=0.7)"]

songs["duration_band"] = pd.Categorical(songs["duration_min"].map(duration_band), categories=DURATION_ORDER, ordered=True)
songs["tempo_band"] = pd.Categorical(songs["tempo"].map(tempo_band), categories=TEMPO_ORDER, ordered=True)
songs["energy_band"] = pd.Categorical(songs["energy"].map(energy_band), categories=ENERGY_ORDER, ordered=True)

for kolom in ["duration_band", "tempo_band", "energy_band"]:
    print(kolom, "->", dict(songs[kolom].value_counts().sort_index()))

duration_band -> {'< 3:30': np.int64(620), '3:30-4:30': np.int64(1055), '4:30-5:30': np.int64(221), '> 5:30': np.int64(30)}
tempo_band -> {'Santai (<90 BPM)': np.int64(228), 'Sedang (90-120 BPM)': np.int64(716), 'Cepat (120-150 BPM)': np.int64(729), 'Sangat cepat (>=150 BPM)': np.int64(253)}
energy_band -> {'Rendah (<0.4)': np.int64(58), 'Sedang (0.4-0.7)': np.int64(714), 'Tinggi (>=0.7)': np.int64(1154)}


## 5. `mood` — kuadran valence × energy (fitur andalan dashboard)

Spotify menyediakan dua fitur yang menggambarkan **emosi** sebuah lagu:

- `valence` (0–1): seberapa "bahagia/cerah" lagu terdengar
- `energy` (0–1): seberapa "keras/intens" lagu terdengar

Menggabungkan keduanya dalam kuadran klasik di psikologi musik:

| | energy **tinggi** | energy **rendah** |
|---|---|---|
| valence **tinggi** | Bahagia & bersemangat | Tenang & hangat |
| valence **rendah** | Tegang & agresif | Melankolis |

Ambang 0.5 (titik tengah rentang) — sederhana dan mudah dijelaskan ke pemirsa.

In [5]:
def mood(valence, energy):
    if valence >= 0.5 and energy >= 0.5: return "Bahagia & bersemangat"
    if valence < 0.5 and energy >= 0.5:  return "Tegang & agresif"
    if valence >= 0.5 and energy < 0.5:  return "Tenang & hangat"
    return "Melankolis"

MOOD_ORDER = ["Bahagia & bersemangat", "Tegang & agresif", "Tenang & hangat", "Melankolis"]
songs["mood"] = pd.Categorical(
    [mood(v, e) for v, e in zip(songs["valence"], songs["energy"])],
    categories=MOOD_ORDER, ordered=True,
)

print(songs["mood"].value_counts().sort_index().to_string())

mood
Bahagia & bersemangat    1099
Tegang & agresif          653
Tenang & hangat            52
Melankolis                122


## 6. `explicit_int` & `is_outlier`

- `explicit_int`: versi 0/1 dari boolean — bisa ikut dihitung dalam korelasi.
- `is_outlier`: penanda nilai ekstrem (di luar 1.5× rentang antarkuartil) untuk
  `duration_min` dan `popularity`. **Sekadar penanda**, bukan penghapusan: lagu
  sangat pendek/panjang tetap fakta yang menarik untuk divisualkan.

In [6]:
songs["explicit_int"] = songs["explicit"].astype(int)

def di_luar_iqr(seri):
    q1, q3 = seri.quantile([0.25, 0.75])
    iqr = q3 - q1
    batas_bawah, batas_atas = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return (seri < batas_bawah) | (seri > batas_atas)

songs["is_outlier"] = di_luar_iqr(songs["duration_min"]) | di_luar_iqr(songs["popularity"])

print("Lagu dengan durasi ekstrem :", int(di_luar_iqr(songs["duration_min"]).sum()))
print("Lagu dengan popularity ekstrem:", int(di_luar_iqr(songs["popularity"]).sum()))
print("Total ditandai outlier      :", int(songs["is_outlier"].sum()), f"({songs['is_outlier'].mean():.1%})")

Lagu dengan durasi ekstrem : 68
Lagu dengan popularity ekstrem: 187
Total ditandai outlier      : 253 (13.1%)


## 7. Tabel `genres_long` — satu baris = satu lagu × satu genre

Alat untuk analisis **per genre** (mis. "rata-rata energy genre metal"). Karena
satu lagu bisa punya beberapa genre, ia muncul lebih dari sekali di tabel ini —
itu memang tujuannya. Untuk hitungan "jumlah lagu" tetap pakai tabel `songs`
(`primary_genre`), supaya tidak dobel hitung.

In [7]:
genres_long = (
    songs.assign(genre=songs["genre"].str.split(","))
    .explode("genre")
)
genres_long["genre"] = genres_long["genre"].str.strip()
genres_long = genres_long[genres_long["genre"] != ""]
# genre_list tidak berguna lagi di tabel ini: setiap baris sudah tepat satu genre
genres_long = genres_long.drop(columns=["genre_list"])

print("Baris songs :", len(songs))
print("Baris genres_long:", len(genres_long), "(lebih banyak karena multi-genre)")
print("\nJumlah lagu per genre (top 10):")
print(genres_long["genre"].value_counts().head(10).to_string())

Baris songs : 1926
Baris genres_long: 3562 (lebih banyak karena multi-genre)

Jumlah lagu per genre (top 10):
genre
pop                 1568
hip hop              746
R&B                  433
Dance/Electronic     377
rock                 225
metal                 64
latin                 63
Unknown               22
Folk/Acoustic         20
country               20


## 8. Tabel `artists` — agregat per artis

Meringkas tiap artis: berapa lagu, seberapa populer rata-ratanya, lagu terpopuler,
genre utama (genre yang paling sering jadi primary_genre), dan rentang tahun aktif.

In [8]:
def modus(seri):
    hitung = seri.value_counts()
    return hitung.index[0] if not hitung.empty else None

artists = songs.groupby("artist").agg(
    n_songs=("song", "count"),
    avg_popularity=("popularity", "mean"),
    best_popularity=("popularity", "max"),
    avg_energy=("energy", "mean"),
    avg_danceability=("danceability", "mean"),
    n_popularity_missing=("is_popularity_missing", "sum"),
    first_year=("year", "min"),
    last_year=("year", "max"),
).reset_index()

# modus tidak bisa langsung di agg di semua versi pandas, jadi dihitung terpisah
main_genre = songs["primary_genre"].groupby(songs["artist"]).agg(modus).rename("main_genre")
active_decade = songs["decade"].astype(str).groupby(songs["artist"]).agg(modus).rename("active_decade")
artists = artists.merge(main_genre, on="artist").merge(active_decade, on="artist")

artists["avg_popularity"] = artists["avg_popularity"].round(1)
artists["avg_energy"] = artists["avg_energy"].round(3)

print("Jumlah artis:", len(artists))
print("\n5 artis paling sering muncul:")
print(artists.nlargest(5, "n_songs")[["artist", "n_songs", "avg_popularity", "main_genre", "active_decade"]].to_string(index=False))

Jumlah artis: 835

5 artis paling sering muncul:
        artist  n_songs  avg_popularity main_genre active_decade
       Rihanna       23            72.3    hip hop        2000an
         Drake       21            67.8    hip hop        2010an
 Calvin Harris       20            67.8    hip hop        2010an
        Eminem       20            73.6    hip hop        2000an
Britney Spears       19            63.1        pop        2000an


## 9. Simpan — CSV **dan** joblib

Catatan teknis: kolom `genre_list` bertipe **list**, tidak bisa disimpan rapi ke CSV.
Oleh karena itu:

- **CSV** (`songs_final.csv`) menyimpan versi tanpa `genre_list` (informasinya sudah
  ada di kolom `genre` dan `primary_genre`) — aman dibuka di Excel.
- **joblib** (`spotify_clean.joblib`) menyimpan versi **lengkap** termasuk `genre_list`
  dan tipe kategori — inilah sumber tunggal untuk dashboard.

In [9]:
# CSV: tanpa kolom list (karena CSV tidak punya tipe list)
songs_csv = songs.drop(columns=["genre_list"])
songs_csv.to_csv(SONGS_OUT, index=False)
genres_long.to_csv(GENRES_OUT, index=False)
artists.to_csv(ARTISTS_OUT, index=False)

# joblib: versi lengkap, satu file untuk semua tabel
data = {"songs": songs, "genres_long": genres_long, "artists": artists}
joblib.dump(data, JOBLIB_OUT)

print("Disimpan:")
for path in [SONGS_OUT, GENRES_OUT, ARTISTS_OUT, JOBLIB_OUT]:
    print(f"  {path}  ({os.path.getsize(path)/1024:.0f} KB)")

Disimpan:
  ../data/clean/songs_final.csv  (441 KB)
  ../data/clean/genres_long.csv  (790 KB)
  ../data/clean/artists.csv  (50 KB)
  ../data/clean/spotify_clean.joblib  (1446 KB)


## 10. Verifikasi: muat ulang dan periksa isi

Penting memastikan berkas benar-benar bisa dibaca lagi (bukan cuma tersimpan) dan
tipe data (kategori berurutan) tetap terjaga setelah round-trip.

In [10]:
data = joblib.load(JOBLIB_OUT)
assert set(data.keys()) == {"songs", "genres_long", "artists"}
songs2, genres2, artists2 = data["songs"], data["genres_long"], data["artists"]

assert list(songs2["decade"].cat.categories) == DECADE_ORDER
assert list(songs2["mood"].cat.categories) == MOOD_ORDER
assert songs2["is_outlier"].dtype == bool
assert isinstance(songs2["genre_list"].iloc[0], list), "genre_list harus list sungguhan!"
assert (songs2["primary_genre"] != "").all()
assert artists2["n_songs"].sum() == len(songs2)

print("Bentuk songs     :", songs2.shape)
print("Bentuk genres_long:", genres2.shape)
print("Bentuk artists   :", artists2.shape)
print("\nKolom baru pada songs:", [c for c in songs2.columns if c not in pd.read_csv(CLEAN_IN).columns])
print("\nContoh 3 lagu dengan fitur baru:")
print(songs2[["song", "primary_genre", "decade", "mood", "duration_band", "tempo_band"]].head(3).to_string(index=False))
print("\nSemua verifikasi LULUS OK")

Bentuk songs     : (1926, 32)
Bentuk genres_long: (3562, 31)
Bentuk artists   : (835, 11)



Kolom baru pada songs: ['key_name', 'mode_name', 'decade', 'duration_band', 'tempo_band', 'energy_band', 'mood', 'explicit_int', 'is_outlier']

Contoh 3 lagu dengan fitur baru:
           song primary_genre decade                  mood duration_band          tempo_band
Sweater Weather          rock 2010an      Tegang & agresif     3:30-4:30 Cepat (120-150 BPM)
   Another Love           pop 2010an      Tegang & agresif     3:30-4:30 Cepat (120-150 BPM)
     Without Me       hip hop 2000an Bahagia & bersemangat     4:30-5:30 Sedang (90-120 BPM)

Semua verifikasi LULUS OK


## 11. Ringkasan & langkah berikutnya

**Artefak yang dihasilkan:**

| File | Isi |
|---|---|
| `data/clean/songs_clean.csv` | hasil cleaning (notebook 02) |
| `data/clean/songs_final.csv` | cleaning + fitur per lagu |
| `data/clean/genres_long.csv` | 1 baris = 1 lagu × 1 genre |
| `data/clean/artists.csv` | agregat per artis |
| `data/clean/spotify_clean.joblib` | ketiga tabel di atas — **sumber dashboard** |

**Fitur yang tersedia untuk analisis:** `key_name`, `mode_name`, `decade`,
`duration_band`, `tempo_band`, `energy_band`, `mood`, `explicit_int`, `is_outlier`,
plus tabel `genres_long` & `artists`.

**Berikutnya (notebook 04):** EDA — distribusi, profil audio per genre, tren per
dekade, scatter mood, **heatmap korelasi (Pearson + Spearman)**, dan rangkuman
insight yang akan diangkat ke dashboard.

## 12. Cerita tahap ini: "membuat data bisa bercerita"

**Cerita tahap ini:**

> *Hook:* "Penonton tidak paham arti 'energy 0.83'. Tapi mereka langsung paham
> kategori **'Tinggi (≥0.7)'** — dan itu membuat grafik bisa dibaca dalam 3 detik,
> bukan 10 detik."
>
> *Angka:* "9 fitur baru dibuat; 3.562 baris di `genres_long` (dari 1.926 lagu);
> 835 artis diringkas jadi 1 baris per artis."
>
> *Makna:* "Fitur turunan adalah **jembatan antara data dan pembaca**. Tanpa `decade`
> tidak ada cerita tren, tanpa `mood` tidak ada cerita emosi, tanpa `duration_band`
> tidak ada cerita 'lagu makin pendek'."

**Tiga "alat bercerita" utama yang kita buat:**

| Alat | Pertanyaan yang dijawab |
|---|---|
| `decade` | "Bagaimana suara musik berubah dari waktu ke waktu?" |
| `mood` (valence × energy) | "Lagu populer itu bahagia atau tegang? Separah apa?" |
| `duration_band` / `tempo_band` / `energy_band` | "Sebagian besar lagu sebentar, sekena apa?" |

**Tips untuk tugas analitismu berikutnya:** setiap kali membuat kolom baru, tanyakan
*"grafik ini akan dibaca orang dalam berapa detik?"* — kategori membuat orang paham
cepat; angka mentah membuat mereka menghitung sendiri.